# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library and Python tools.

### Dataset Source
The dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's discover the available record sets and their schema using their `@id`. We also view the fields in each record set. All access is by Croissant `@id`.

In [ ]:
# List all record sets and their fields by @id
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    record_sets = dataset.metadata.record_sets
else:
    # Fallback for metadata where record sets may be in a different attribute
    record_sets = getattr(dataset.metadata, 'recordSet', [])

if not record_sets:
    # Try to autodiscover via dataset API (most datasets place record sets here)
    record_sets = [r for r in dataset.record_sets]

record_set_ids = []

print("Record Sets and Fields: (all referenced by @id)")
for rs in record_sets:
    # Some Croissant schemas store record sets as dicts, others as objects. Try both:
    rs_obj = rs
    if isinstance(rs, dict):
        rs_id = rs.get('@id', None)
        rs_name = rs.get('name', '(no name)')
        fields = rs.get('field', [])
    else:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', '(no name)')
        fields = getattr(rs, 'field', [])

    if not rs_id:
        continue
    record_set_ids.append(rs_id)
    print(f"- recordSet: {rs_id}  name: {rs_name}")
    # Print fields for this record set
    if fields:
        for f in fields:
            field_id = None
            field_name = '(no name)'
            if isinstance(f, dict):
                field_id = f.get('@id', None)
                field_name = f.get('name', '(no name)')
            else:
                field_id = getattr(f, '@id', None)
                field_name = getattr(f, 'name', '(no name)')
            print(f"    - field: {field_id}  name: {field_name}")
    else:
        print("    (No fields described)")

if not record_set_ids:
    # Try using the mlcroissant dataset API for listing IDs
    record_set_ids = [rs['@id'] for rs in dataset.to_json().get('recordSet', [])]
    print("recordSet @ids:", record_set_ids)

# If still empty, try to discover via dataset.record_sets
if not record_set_ids and hasattr(dataset, 'record_sets'):
    record_set_ids = list(dataset.record_sets.keys())
    print("recordSet @ids:", record_set_ids)

# Show once for user
print("\nAll discovered recordSet @ids:")
for rsid in record_set_ids:
    print(f"  {rsid}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. All access uses the record set and field `@id`. We'll extract all record sets we found.

In [ ]:
# Extract data from all discovered record sets by @id
dataframes = {}

print("Loading data from record sets...")
for record_set_id in record_set_ids:
    print(f"Extracting records from recordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  {len(df)} rows, {len(df.columns)} columns")
    except Exception as e:
        print(f"  Could not extract records from {record_set_id}: {e}")

if dataframes:
    # Use the first record set for demonstration
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nSample columns from recordSet @id: {first_record_set_id}")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing, and grouping.

We'll demonstrate this using a numeric field (change the field `@id` as needed).

In [ ]:
# Find a numeric field in the first record set
import numpy as np

target_rs_id = first_record_set_id
df = dataframes[target_rs_id]

# Try to infer a numeric column (@id) for demo (commonly age or year, etc.)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try to coerce some field (e.g., contains 'age', 'interval', 'year')
    for col in df.columns:
        if any(substr in col.lower() for substr in ['age', 'interval', 'year', 'count']):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().sum() > 0:
                    numeric_field_id = col
                    break
            except Exception:
                continue

if numeric_field_id is not None:
    print(f"Using numeric field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a categorical field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field = col
            break

    if group_field is not None:
        print(f"\nGrouped means of {numeric_field_id} by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")
else:
    print("No numeric field found for EDA. Please check the dataset fields.")

## 5. Visualization
Visualize the numeric field distribution and, if possible, its relationship with a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if a numeric field is present
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load the FAIR² dataset using its Croissant schema, explored its available record sets and fields by `@id`, and demonstrated data extraction, filtering, normalization, grouping, and visualization. These steps provide a systematic foundation for deeper analysis and modeling on complex, FAIR-compliant datasets.

For further research, consider exploring dataset documentation and the detailed data dictionary available via the Croissant metadata.